# Load open mine production CSVs into bronze

Reads every CSV of the open mine production database from the default lakehouse `Files` area and writes one Delta table per file to `dbo` (table name = file name, columns lower snake_case).

Upload the CSVs from `docs/open_database_mine_production/data` to `Files/open_database_mine_production/data` first. `facilities.gpkg` and `facilities_text.txt` are not loaded; `facilities.csv` already holds the geometry as WKT in `geom`.


In [ ]:
import re

SOURCE_DIR = "Files/open_database_mine_production/data"
SCHEMA = "dbo"
TABLES = [
    "capacity",
    "coal",
    "commodities",
    "facilities",
    "material_ids",
    "minerals",
    "ownership",
    "processing",
    "reserves",
    "source_ids",
    "transport",
    "waste",
]


In [ ]:
def clean_column(name: str) -> str:
    """Return a lower snake_case column name."""
    return re.sub(r"\W+", "_", name.strip()).strip("_").lower()


results = {}
for table in TABLES:
    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{SOURCE_DIR}/{table}.csv")
    )
    df = df.toDF(*[clean_column(c) for c in df.columns])
    df.write.mode("overwrite").format("delta").option("overwriteSchema", True).saveAsTable(f"{SCHEMA}.{table}")
    results[table] = df.count()

print(results)
